# Patents linked to publications
_(Created in June 2019)_

This notebook is dependent on the following libraries

In [1]:
import dimcli
import pandas as pd
from pandas.io.json import json_normalize
#import plotly_express as px
#from plotly.offline import init_notebook_mode # needed for exports 
#init_notebook_mode(connected=True)
import time
import dimcli
from dimcli.shortcuts import dslquery, dslqueryall, chunks_of, normalize_key
dimcli.login()

DimCli v0.6.1 - Succesfully connected to <https://app.dimensions.ai> (method: dsl.ini file)


___
# 1. Data Extraction and Preparation

In [2]:
GRIDID = "grid.13097.3c"


In [3]:
df_all_pubs_per_year = dslquery(f"""search publications where research_orgs.id="{GRIDID}" return year limit 1000""").as_dataframe()



Returned Year: 182


In [4]:

# Get full list of reelvant publications linked to this organization
pubs_details = dslqueryall(f"""search publications where research_orgs.id="{GRIDID}" and year in [2010:2017] return publications[basics-author_affiliations]""")
df_pubs_details = pubs_details.as_dataframe()



1000 / 46189
2000 / 46189
3000 / 46189
4000 / 46189
5000 / 46189
6000 / 46189
7000 / 46189
8000 / 46189
9000 / 46189
10000 / 46189
11000 / 46189
12000 / 46189
13000 / 46189
14000 / 46189
15000 / 46189
16000 / 46189
17000 / 46189
18000 / 46189
19000 / 46189
20000 / 46189
21000 / 46189
22000 / 46189
23000 / 46189
24000 / 46189
25000 / 46189
26000 / 46189
27000 / 46189
28000 / 46189
29000 / 46189
30000 / 46189
31000 / 46189
32000 / 46189
33000 / 46189
34000 / 46189
35000 / 46189
36000 / 46189
37000 / 46189
38000 / 46189
39000 / 46189
40000 / 46189
41000 / 46189
42000 / 46189
43000 / 46189
44000 / 46189
45000 / 46189
46000 / 46189
46189 / 46189


Create a new list of publications with simplifed FOR data

In [5]:
# Create a new list of publications with simplifed FOR data
# Ensure that all pubs have a valid (empty, even) FOR value, also remove the FOR digit prefix to improve legibility
for x in pubs_details.publications:
    if not 'category_for' in x:
        x['category_for'] = ""
    else:
        x['category_for'] = [{'name' : x['name'][5:]} for x in x['category_for']] 
df_pubs_for = json_normalize(pubs_details.publications, record_path=['category_for'], meta=["id", "type", "category_for", ["journal", "title"], "year"], errors='ignore', record_prefix='for_')





Get the patents infos using the publications 

In [6]:
# query structure is:
# d=dslquery(f"""search patents where publication_ids in ["pub.1111511314","pub.1113174788","pub.1111902055"] return patents limit 1000""")

from itertools import islice
def chunks_of(data, size):
    it = iter(data)
    chunk = list(islice(it, size))
    while chunk:
        yield chunk
        chunk = list(islice(it, size))

SIZE = 400

def run(ids_list):
    patents_out, n  = [], 0
    for chunk in chunks_of(ids_list, SIZE): # chunks of 200 args
        n += 1
        temp = ','.join(['"{}"'.format(i) for i in chunk])
        data = dslquery(f"""search patents where publication_ids in [{temp}] return patents[basics+publication_ids+id+FOR + RCDC +times_cited] limit 1000""")
        patents_out += data.patents
        print("[log] ", n*SIZE, " pubs > patents: ", len(data.patents))
        time.sleep(1)
    return patents_out
        
patents_list = run(list(df_pubs_details['id']))

Returned Patents: 0
WARNINGS [1]
Field 'RCDC' is deprecated in favor of category_rcdc. Please refer to https://docs.dimensions.ai/dsl/releasenotes.html for more details
[log]  400  pubs > patents:  0
Returned Patents: 0
WARNINGS [1]
Field 'RCDC' is deprecated in favor of category_rcdc. Please refer to https://docs.dimensions.ai/dsl/releasenotes.html for more details
[log]  800  pubs > patents:  0
Returned Patents: 0
WARNINGS [1]
Field 'RCDC' is deprecated in favor of category_rcdc. Please refer to https://docs.dimensions.ai/dsl/releasenotes.html for more details
[log]  1200  pubs > patents:  0
Returned Patents: 0
WARNINGS [1]
Field 'RCDC' is deprecated in favor of category_rcdc. Please refer to https://docs.dimensions.ai/dsl/releasenotes.html for more details
[log]  1600  pubs > patents:  0
Returned Patents: 2 (total = 2)
WARNINGS [1]
Field 'RCDC' is deprecated in favor of category_rcdc. Please refer to https://docs.dimensions.ai/dsl/releasenotes.html for more details
[log]  2000  pubs

After going through all publications and extracting related patents, let's save the patents data so that we can use it later:

In [7]:
df_patent_details = pd.DataFrame().from_dict(patents_list)
# save to CSV
df_patent_details.to_csv("data/sydney_2019_patents_by_id.csv")
# display top 3 rows
df_patent_details.head(3)

,publication_ids,publication_date,FOR,inventor_names,id,year,filing_status,assignee_names,assignees,title,RCDC,granted_year,times_cited
0,"[pub.1019581843, pub.1092164786]",2018-01-24,"[{'id': '2746', 'name': '0801 Artificial Intel...","[ZIGON, ROBERT, VANWINKLE, RACHEL, SMITH, JERE...",EP-2347352-A4,2009,Application,"[Beckman Coulter Inc, BECKMAN COULTER INC]","[{'id': 'grid.418254.e', 'name': 'Beckman Coul...",INTERACTIVE TREE PLOT FOR FLOW CYTOMETRY DATA,NaN,NaN,NaN
1,"[pub.1030319901, pub.1034748881, pub.103717905...",2018-04-17,"[{'id': '2464', 'name': '0305 Organic Chemistr...","[Roberto Motterlini, Roberta FORESTI, Thierry ...",US-9944669-B2,2015,Grant,[Centre National de la Recherche Scientifique ...,"[{'id': 'grid.7429.8', 'acronym': 'INSERM', 'n...","Fumarate-CO-releasing molecule hybrids, their ...","[{'id': '501', 'name': 'Brain Disorders'}]",2018.0,NaN
2,[pub.1091248898],2016-04-07,NaN,"[Yukiko Kubota, Timothy John Klemmer, Kai Chie...",US-20160099016-A1,2015,Application,"[Seagate Technology LLC, SEAGATE TECHNOLOGY LLC]","[{'id': 'grid.462839.2', 'name': 'Seagate (Uni...",MAGNETIC STACK INCLUDING MgO-Ti(ON) INTERLAYER,NaN,NaN,NaN


We also want to normalise the assignees data in order to analyse it further later on (ps: first we need to make the assigness data structure more regular)

In [8]:
for x in patents_list:
    if not 'assignees' in x:
        x['assignees'] = []
df_patents_assignees = json_normalize(patents_list, record_path=['assignees'], meta=['id', 'year', 'title'], meta_prefix="grant_")
# save to CSV 
df_patents_assignees.to_csv("data/sydney_2019_patents_by_assignees1.csv")
df_patents_assignees.head()

,id,name,country_name,acronym,grant_id,grant_year,grant_title
0,grid.418254.e,Beckman Coulter (United States),United States,NaN,EP-2347352-A4,2009,INTERACTIVE TREE PLOT FOR FLOW CYTOMETRY DATA
1,grid.7429.8,French Institute of Health and Medical Research,France,INSERM,US-9944669-B2,2015,"Fumarate-CO-releasing molecule hybrids, their ..."
2,grid.410511.0,Paris 12 Val de Marne University,France,UPEC,US-9944669-B2,2015,"Fumarate-CO-releasing molecule hybrids, their ..."
3,grid.4444.0,French National Centre for Scientific Research,France,CNRS,US-9944669-B2,2015,"Fumarate-CO-releasing molecule hybrids, their ..."
4,grid.462839.2,Seagate (United States),United States,NaN,US-20160099016-A1,2015,MAGNETIC STACK INCLUDING MgO-Ti(ON) INTERLAYER


Let's do the same type of normalization for FOR codes 

In [9]:
for x in patents_list:
    if not 'FOR' in x:
        x['FOR'] = ""
    else:
        x['FOR'] = [{'name' : x['name'][5:]} for x in x['FOR']] 
df_patents_for = json_normalize(patents_list, record_path=['FOR'], meta=["id", "year", "title"], errors='ignore', record_prefix='for_')
# save to CSV 
df_patents_for.to_csv("data/sydney_2019_patents_by_FOR.csv")
df_patents_for.head()

,for_name,id,year,title
0,Artificial Intelligence and Image Processing,EP-2347352-A4,2009,INTERACTIVE TREE PLOT FOR FLOW CYTOMETRY DATA
1,Organic Chemistry,US-9944669-B2,2015,"Fumarate-CO-releasing molecule hybrids, their ..."
2,Other Physical Sciences,US-8143254-B2,2006,Methods for modulating ion channels
3,Other Physical Sciences,WO-2018024813-A1,2017,IMAGING TECHNOLOGIES
4,Organic Chemistry,US-10023570-B2,2016,"Substituted pyrazolo[1,5-A]pyridine compounds ..."


Finally, let's create another publications index, including only the KSU publications cited by the patents extracted above. 

In [10]:
# extract pubs from patents list
pubs_referenced_from_patents = []
for x in patents_list:
    if x['publication_ids']:
        pubs_referenced_from_patents += x['publication_ids']
# remove duplicates
pubs_referenced_from_patents = list(set(pubs_referenced_from_patents))
len(pubs_referenced_from_patents)

37228

Intersect list of publications from KSU with list of publications mentioned in patents

In [11]:
df_linked_pubs =  df_pubs_details[df_pubs_details['id'].isin(pubs_referenced_from_patents)]
df_linked_pubs.reset_index(drop=True)
# save to CSV 
df_linked_pubs.to_csv("data/sydney_pubs_linked_to_patents.csv")
df_linked_pubs.head()


,year,issue,id,pages,type,volume,title,journal.id,journal.title
1610,2017,Trends Pharmacol. Sci. 34 2013,pub.1090348791,444-451,article,13,NRF2 deficiency replicates transcriptomic chan...,jour.1048218,Redox Biology
1813,2017,10,pub.1092164786,1584-1797,article,47,Guidelines for the use of flow cytometry and c...,jour.1054998,European Journal of Immunology
2021,2017,35,pub.1091248898,29857-29862,article,9,Titanium Oxynitride Thin Films with Tunable Do...,jour.1041450,ACS Applied Materials & Interfaces
2403,2017,NaN,pub.1084749935,11-23,article,87,Cardiac voltage-gated ion channels in safety p...,jour.1102215,Journal of Pharmacological and Toxicological M...
3861,2017,6,pub.1070928630,891-898,article,58,Intraoperative Assessment of Tumor Resection M...,jour.1012368,Journal of Nuclear Medicine


Also, create a version of `df_linked_pubs` with simplified FOR codes so that it's easier to visualise.

In [12]:
df_linked_pubs_for =  df_pubs_for[df_pubs_for['id'].isin(pubs_referenced_from_patents)]
df_linked_pubs_for.reset_index(drop=True)
df_linked_pubs_for.head()


,id,type,category_for,journal.title,year


In [13]:
for x in pubs_referenced_from_patents:
     if not 'research_orgs' in x:
        x['research_orgs'] = []
df_linked_orgs = json_normalize(pubs_referenced_from_patents, record_path=['research_orgs'], meta=['researchers', 'year', 'title'], meta_prefix="org_")
# save to CSV 
df_linked_orgs.to_csv("data/agritech_2019_patents_by_orgs.csv")
df_linked_orgs.head()

TypeError: 'str' object does not support item assignment

---
#### That's it - now it's time to create some visualizations!
___

# 2. Data Analysis

## 2.1 Publications

In [ ]:
px.bar(df_all_pubs_per_year, x="id", y="count", title="Total Publications per year from KSU")

In [ ]:
px.scatter(df_pubs_for, x="year", y="for_name", color="type", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of pubs in last 10 years (marginal subplots = X/Y totals)")

In [ ]:
px.bar(df_linked_pubs.groupby('year',  as_index=False).count(), x="year", y="id", title="Publications mentioned in patents, by year of publication")

In [ ]:
px.scatter(df_linked_pubs_for, x="year", y="for_name", color="type", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of pubs mentioned in patents (marginal subplots = X/Y totals)")

## 2.2 Patents

In [ ]:
px.bar(df_patent_details.groupby('year',  as_index=False).count(),  x="year", y="id",  title="Patents per Year citing publications from KSU")

In [ ]:
px.scatter(df_patents_for, x="year", y="for_name", hover_name="for_name", marginal_x="histogram", marginal_y="histogram", height=800, title="Research areas of patents (marginal subplots = X/Y totals)")

In [ ]:
px.scatter(df_patent_details, x="year", y="times_cited", hover_name="title",  hover_data=['id'], facet_col="filing_status", title="Patents per Year VS Timed Cited VS Filing Status")

## 2.3 Assignees of patents

In [ ]:
px.bar(df_patents_assignees.groupby('name',  as_index=False).count().sort_values(by="grant_id", ascending=False),  x="name", y="grant_id", hover_name="name",  height=400,  title="Assignees by No of Patents")

In [ ]:
px.scatter(df_patents_assignees,  x="grant_year", y="name", color="country_name", hover_name="name",  hover_data=["id"],  height=800, title="Assignees By Country and Year")